# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [97]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_books(min_rating, max_price): #min_rating, max_price
    
    url = "https://books.toscrape.com"
    user_agent = { "User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"}

    #get response
    response = requests.get(url, headers = user_agent)
    #response.status_code

    #step 2: parse the html text
    soup = BeautifulSoup(response.text, "html.parser")

    #1. Extracting data
    #get UPC: The Universal Product Code (UPC) of the book.
    books = soup.find_all("article", "product_pod")


    book_names = [book.find("h3").find("a").get("title").strip() for book in books]
    book_av = [book.find("p", class_ = "instock availability").text.strip() for book in books]
    book_prices = [book.find("p", class_ = "price_color").text.strip("Â£") for book in books]
    #get the full url
    #book_hrefs = [url+ "/" + book.find("h3").find("a").get("href").strip() for book in books]
    
    #tags = [book.find("p", class_ = apply(lambda x: ) for book in books]
    ratings = [book.select_one("p")["class"][1].strip() for book in books]
    #book_ratings = [tag[ for tag in tags]

   
    ##########################################################################
    # Looping through next pages to get: UPC and book description
    ##########################################################################
    #get the full url
    book_hrefs = [url+ "/" + book.find("h3").find("a").get("href").strip() for book in books]
    books_description = []
    upc = []
    for pag in book_hrefs:
        #get the html page
        response_pag = requests.get(pag)
        #get the parsed html page
        soup_pag = BeautifulSoup(response_pag.content, "html.parser")
        #Extract data
        book_pag = soup_pag.find("article", class_="product_page")
        description_div = book_pag.find("div", id = "product_description")
        book_desc = description_div.find_next_sibling("p").text
        book_upc = book_pag.select_one("tr td").text
        books_description.append(book_desc)
        upc.append(book_upc)

     #########################################################################
    # Creating dataframe
    ##########################################################################
    df = pd.DataFrame(
                        {"Book Name": book_names,
                         "Book Availability": book_av,
                         "Book Price (in £)" : book_prices,
                         "Rating": ratings,
                         "UPC": upc,
                         "Description": books_description
                        }
                      )
    ##########################################################################
    # Data wrangling
    ##########################################################################
    #Convert book price to folat
    df["Book Price (in £)"] = df["Book Price (in £)"].apply(float)
    
    #map text number to numerals:
    text_to_num = {
                    'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
                    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
                  }
    df["Rating"] = df.Rating.str.lower().map(text_to_num)
    df["Rating"].apply(int)
    #filter data: min_rating, max_price
    condition = (df["Rating"] >= min_rating) & (df["Book Price (in £)"] <= max_price)
    df = df[condition]
    #condition
    return df 

    
df= scrape_books(0, 40)
print(df.dtypes)
df



Book Name             object
Book Availability     object
Book Price (in £)    float64
Rating                 int64
UPC                   object
Description           object
dtype: object


,Book Name,Book Availability,Book Price (in £),Rating,UPC,Description
5,The Requiem Red,In stock,22.65,1,f77dbf2323deb740,Patient Twenty-nine.A monster roams the halls ...
6,The Dirty Little Secrets of Getting Your Dream...,In stock,33.34,4,2597b5a345f45e1b,Drawing on his extensive experience evaluating...
7,The Coming Woman: A Novel Based on the Life of...,In stock,17.93,3,e72a5dfc7e9267b2,"""If you have a heart, if you have a soul, Kare..."
8,The Boys in the Boat: Nine Americans and Their...,In stock,22.60,4,e10e1e165dc8be4a,For readers of Laura Hillenbrand's Seabiscuit ...
10,"Starving Hearts (Triangular Trade Trilogy, #1)",In stock,13.99,2,0312262ecafa5a40,"Since her assault, Miss Annette Chetwynd has b..."
11,Shakespeare's Sonnets,In stock,20.66,4,30a7f60cd76ca58c,This book is an important and complete collect...
12,Set Me Free,In stock,17.46,5,ce6396b0f23f6ecc,Aaron Ledbetter’s future had been planned out ...
14,Rip it Up and Start Again,In stock,35.02,5,a34ba96d4081e6a4,"Punk's raw power rejuvenated rock, but by the ..."
16,Olio,In stock,23.88,1,feb7cc7701ecf901,"Part fact, part fiction, Tyehimba Jess's much ..."
17,Mesaerion: The Best Science Fiction Stories 18...,In stock,37.59,1,e30f54cea9b38190,"Andrew Barger, award-winning author and engine..."
